# L33 Verification Debug Notebook

This notebook contains a deliberately flawed single-server queue. The goal is to practice verification: read the conceptual model, run small traces, write tests, and repair the implementation.

## Conceptual Model

- Arrivals follow a Poisson process with rate `lam`.
- Service times are exponential with rate `mu`.
- One FCFS server handles customers.
- The primary output is mean waiting time in queue, `Wq`.

For a stable M/M/1 queue, the analytical benchmark is

$$W_q = \frac{\rho}{\mu - \lambda}, \qquad \rho=\lambda/\mu < 1.$$

In [ ]:
import math

import numpy as np
import pandas as pd
import simpy

In [ ]:
def flawed_mm1(lam=0.8, mu=1.0, sim_time=2000, seed=123, trace=False):
    """Run a deliberately flawed M/M/1 model for verification practice."""
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    server = simpy.Resource(env, capacity=1)
    stats = {
        "arrivals": 0,
        "completions": 0,
        "waits": [],
        "trace": [],
    }

    def log(customer_id, event, service_time=None):
        if trace:
            stats["trace"].append(
                {
                    "time": env.now,
                    "customer_id": customer_id,
                    "event": event,
                    "queue_len": len(server.queue),
                    "in_service": server.count,
                    "service_time": service_time,
                }
            )

    def customer(customer_id):
        arrival_time = env.now
        service_time = rng.exponential(1 / mu)
        log(customer_id, "arrive", service_time)
        with server.request() as req:
            yield req
            log(customer_id, "start_service", service_time)
            wait = env.now - service_time  # BUG: should use arrival_time.
            stats["waits"].append(wait)
            yield env.timeout(service_time)
            stats["completions"] += 1
            log(customer_id, "depart", service_time)

    def arrivals():
        customer_id = 0
        while env.now < sim_time:
            customer_id += 1
            interarrival = rng.exponential(1 / lam)
            yield env.timeout(interarrival)
            stats["arrivals"] += 1
            env.process(customer(customer_id))

    env.process(arrivals())
    env.run(until=sim_time)
    return {
        "mean_wait_q": float(np.mean(stats["waits"])) if stats["waits"] else 0.0,
        "arrivals": stats["arrivals"],
        "completions": stats["completions"],
        "trace": pd.DataFrame(stats["trace"]),
    }

## Step 1: Run a Small Trace

Use a short run to inspect event order, queue length, and waiting-time logic. Do not start with a long stochastic run.

In [ ]:
small = flawed_mm1(lam=0.8, mu=1.0, sim_time=12, seed=4, trace=True)
small["trace"]

## Step 2: Write Verification Checks

Fill in the expected behavior before changing the model. Good checks should fail for a meaningful reason.

In [ ]:
def expected_mm1_wq(lam, mu):
    rho = lam / mu
    return rho / (mu - lam)


def check_zero_arrivals():
    result = flawed_mm1(lam=0.0, mu=1.0, sim_time=100, seed=10)
    assert result["arrivals"] == 0
    assert result["completions"] == 0
    assert result["mean_wait_q"] == 0.0


def check_against_mm1_formula():
    lam = 0.6
    mu = 1.0
    observed = flawed_mm1(lam=lam, mu=mu, sim_time=50_000, seed=22)["mean_wait_q"]
    expected = expected_mm1_wq(lam, mu)
    assert math.isclose(observed, expected, rel_tol=0.20), (observed, expected)

In [ ]:
# Run the checks one at a time. The failures are evidence, not a problem.
# check_zero_arrivals()
# check_against_mm1_formula()

## Step 3: Repair and Document

Create `corrected_mm1` below. Keep the same return dictionary so the checks can be reused.

Minimum repair targets:

1. Waiting time in queue is computed from arrival time.
2. Zero-arrival case terminates cleanly.
3. The arrival process does not schedule customers after the run horizon.
4. Analytical benchmark check passes with a defensible tolerance.

In [ ]:
def corrected_mm1(lam=0.8, mu=1.0, sim_time=2000, seed=123, trace=False):
    # TODO: implement the corrected model by adapting flawed_mm1.
    raise NotImplementedError